# E-commerce Sales & Customer Analytics

End-to-end analysis of a raw e-commerce orders export: cleaning messy data, exploring revenue and customer patterns, and answering business questions.

**Tech stack:** Python (Pandas, NumPy, Matplotlib, Seaborn), SQL (SQLite)

See `scripts/generate_data.py` and `scripts/analysis.py` for the full source. This notebook walks through the same pipeline interactively.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/ecommerce_orders_raw.csv")
df.head()

## 1. Initial data quality check

Before touching the data, check what's actually wrong with it: duplicates, missing values, inconsistent formatting, outliers.

In [ ]:
print("Rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nUnique region values (note casing issues):", df['region'].unique())
print("\nUnique payment methods (note casing issues):", df['payment_method'].unique())

## 2. Cleaning

- Drop exact duplicates
- Standardize text casing (`region`, `payment_method`)
- Remove invalid negative quantities
- Cap unit-price outliers using the IQR method (likely data-entry errors, not real sales)
- Impute missing `customer_age` / `delivery_days` with the median; fill missing `region` with `Unknown`
- Leave missing `rating` as-is rather than fabricating satisfaction scores

In [ ]:
df = df.drop_duplicates()
df['region'] = df['region'].str.strip().str.title()
df['payment_method'] = df['payment_method'].str.strip().str.title()
df = df[df['quantity'] > 0]

q1, q3 = df['unit_price'].quantile([0.25, 0.75])
iqr = q3 - q1
upper_bound = q3 + 3 * iqr
df['unit_price'] = np.where(df['unit_price'] > upper_bound, upper_bound, df['unit_price'])
df['revenue'] = (df['unit_price'] * df['quantity']).round(2)

df['region'] = df['region'].fillna('Unknown')
df['customer_age'] = df['customer_age'].fillna(df['customer_age'].median()).astype(int)
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median()).astype(int)
df['order_date'] = pd.to_datetime(df['order_date'])

print("Clean rows:", len(df))
df.head()

## 3. Revenue by category

In [ ]:
rev_cat = df.groupby('product_category')['revenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x=rev_cat.values, y=rev_cat.index)
plt.title('Total Revenue by Product Category')
plt.xlabel('Revenue (₹)')
plt.show()

## 4. Monthly revenue trend

In [ ]:
df['month'] = df['order_date'].dt.to_period('M').astype(str)
monthly = df.groupby('month')['revenue'].sum()
plt.figure(figsize=(10,5))
monthly.plot(marker='o')
plt.title('Monthly Revenue Trend')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Does delivery speed affect ratings?

In [ ]:
rated = df.dropna(subset=['rating'])
low = rated[rated['rating'] <= 2]['delivery_days'].mean()
high = rated[rated['rating'] >= 4]['delivery_days'].mean()
print(f"Avg delivery days for low ratings (1-2): {low:.1f}")
print(f"Avg delivery days for high ratings (4-5): {high:.1f}")
print("-> No strong relationship found in this dataset" if abs(low-high) < 0.3 else "-> Delivery speed appears linked to rating")

## Key Takeaways

- Electronics is the top revenue-generating category
- East region leads in total revenue
- Average customer rating is ~4.0/5
- Delivery speed shows no strong relationship with rating in this dataset — worth testing with a larger sample before drawing conclusions
- See `sql/queries.sql` for the equivalent business-question analysis done in SQL